# 6.01 Character / Recursive / Token Splitter

가장 기초가 되는 3개의 **general-purpose** 텍스트 분할기를 다룬다. RAG 파이프라인의 첫 번째 전처리 단계이며, 이 선택이 임베딩 품질과 검색 점수를 크게 좌우한다.

## 학습 목표

- `CharacterTextSplitter` 의 단일 separator 기반 단순 분할을 이해한다
- `RecursiveCharacterTextSplitter` 의 **separator 우선순위**(`["\n\n", "\n", " ", ""]`) 동작을 파악한다
- `TokenTextSplitter` 로 **모델 토큰 한도** 에 맞춰 자르는 경로(tiktoken)를 쓴다
- `chunk_size` / `chunk_overlap` 의 실무 기본값과 튜닝 지침을 안다

## 언제 쓰나

- **기본 선택지**: 일반 텍스트/혼합 문서는 `RecursiveCharacterTextSplitter` 하나로 충분
- **토큰 한도 엄격**: 임베딩 모델 컨텍스트 한도에 맞출 때 `TokenTextSplitter`
- **단일 구분자로 충분**: CSV 행, 로그 이벤트 등 단순 구조는 `CharacterTextSplitter`

## 6.01.1 환경 설정

필요 패키지: `langchain-text-splitters`, `tiktoken`. 외부 API 키 **불필요**.

In [ ]:
# !pip install -U langchain-text-splitters tiktoken

from dotenv import load_dotenv
load_dotenv(override=True)

# Splitter 01/02 는 키 불필요 — import 확인만
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
print('import OK')

## 6.01.2 기본 사용법 — 한국어 샘플

5문단·약 400자 한국어 문단을 샘플로 쓴다. 아래 세 splitter 모두 이 샘플을 공유한다.

### `CharacterTextSplitter` — 단일 separator

기본 separator 는 `"\n\n"`(문단 구분). 해당 구분자가 없거나, 구분 후 조각이 `chunk_size` 를 넘으면 **초과를 그대로 허용** 한다(경고 발생). 단순 구조에만 쓴다.

In [ ]:
SAMPLE = (
    "LangChain은 LLM 애플리케이션을 조립하기 위한 프레임워크다. "
    "문서 로더, 임베딩, vector store, 에이전트 등 RAG의 전 과정을 추상화한다.\n\n"
    "텍스트 분할(chunking)은 RAG 품질을 좌우하는 전처리 단계다. "
    "너무 크게 자르면 임베딩이 의미를 희석하고, 너무 작게 자르면 문맥이 끊겨 검색 품질이 떨어진다.\n\n"
    "실무에서는 `RecursiveCharacterTextSplitter` 가 가장 널리 쓰인다. "
    "단락(\\n\\n) → 줄바꿈(\\n) → 문장(. ) → 공백 순으로 재귀적으로 내려가며 "
    "chunk_size 안에 들어오도록 자른다.\n\n"
    "한국어는 영문과 토큰 분포가 달라 tiktoken 기준으로 300–500 토큰, "
    "문자 기준으로 600–1000자가 초기값으로 무난하다. "
    "chunk_overlap 은 chunk_size 의 10–20% 가 안전한 기본값이다.\n\n"
    "요약 · QA · 멀티-홉 질의 등 다운스트림 과제에 따라 "
    "overlap 과 chunk_size 를 실험으로 조정하라."
)
print(f'sample length: {len(SAMPLE)} chars')

char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=200,
    chunk_overlap=30,
    length_function=len,            # 기본 len: 문자 수
)
char_chunks = char_splitter.split_text(SAMPLE)
print(f'[Character] chunks: {len(char_chunks)}')
for i, c in enumerate(char_chunks):
    print(f'  [{i}] {len(c):>3}자 | {c[:40]}…')

### `RecursiveCharacterTextSplitter` — separator 우선순위

기본값 `["\n\n", "\n", " ", ""]` 순서:
1. 먼저 `"\n\n"` 로 잘라보고, 조각이 `chunk_size` 이하면 통과
2. 넘으면 같은 조각을 `"\n"` 로 재분할 → 이후 공백 → 문자 단위

→ **의미 단위를 최대한 보존** 하면서 크기 한도를 맞춘다. 일반 텍스트의 기본값.

In [ ]:
rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],  # 한국어는 마침표+공백 추가
    length_function=len,
)
rec_chunks = rec_splitter.split_text(SAMPLE)
print(f'[Recursive] chunks: {len(rec_chunks)}')
for i, c in enumerate(rec_chunks):
    print(f'  [{i}] {len(c):>3}자 | {c[:40]}…')

## 6.01.3 `TokenTextSplitter` — 토큰 단위 분할

임베딩/LLM 모델의 **실제 토큰 한도** 에 맞춰 잘라야 할 때 쓴다. 내부적으로 `tiktoken`.

| 모델 | 권장 encoding |
|------|---------------|
| `text-embedding-3-*`, `gpt-4o`, `gpt-4.1` | `cl100k_base` |
| `gpt-4` (legacy), `gpt-3.5-turbo` | `cl100k_base` |
| o-series 최신 | `o200k_base` |
| HuggingFace 계열 | `transformers` 토크나이저 (`from_huggingface_tokenizer`) |

파라미터: `encoding_name` 과 `model_name` 중 하나를 쓴다. `model_name` 이 있으면 encoding 이 자동 선택된다. `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)` 팩토리는 **토큰 길이로 재면서 separator 는 재귀 분할** 하는 실무 조합.

In [ ]:
tok_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",   # OpenAI GPT-4 계열 기본
    chunk_size=80,                  # 토큰 수 기준
    chunk_overlap=10,
)
tok_chunks = tok_splitter.split_text(SAMPLE)
print(f'[Token      ] chunks: {len(tok_chunks)}')

rec_tok = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=80,
    chunk_overlap=10,
)
rt_chunks = rec_tok.split_text(SAMPLE)
print(f'[Recur+tok  ] chunks: {len(rt_chunks)}')

## 6.01.4 분할 결과 비교 표

동일 샘플 · 동일 `chunk_size`(문자=200, 토큰=80) · overlap 을 맞춘 경우의 차이.

In [ ]:
rows = [
    ("CharacterTextSplitter",           len(char_chunks),  sum(len(c) for c in char_chunks) // max(len(char_chunks),1)),
    ("RecursiveCharacterTextSplitter",  len(rec_chunks),   sum(len(c) for c in rec_chunks)  // max(len(rec_chunks),1)),
    ("TokenTextSplitter (cl100k)",       len(tok_chunks),   sum(len(c) for c in tok_chunks)  // max(len(tok_chunks),1)),
    ("Recursive + tiktoken",             len(rt_chunks),    sum(len(c) for c in rt_chunks)   // max(len(rt_chunks),1)),
]
print(f'{"splitter":<35} | {"#chunks":>7} | {"avg 문자":>10}')
print('-'*60)
for name, n, avg in rows:
    print(f'{name:<35} | {n:>7} | {avg:>10}')

## 6.01.5 vector store 연동 (한 줄 예)

분할 → `Document` → 임베딩 → `InMemoryVectorStore` → `similarity_search`. 본격적인 내용은 `../03_vectorstores/01_inmemory_and_faiss.ipynb` 참고.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import DeterministicFakeEmbedding

# 외부 키 없이 돌릴 수 있도록 FakeEmbedding 사용 (실무는 OpenAIEmbeddings 등으로 교체)
emb = DeterministicFakeEmbedding(size=128)
docs = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(rec_chunks)]
store = InMemoryVectorStore.from_documents(docs, embedding=emb)
hits = store.similarity_search("chunk_overlap 의 적정 비율은?", k=2)
for h in hits:
    print('-', h.metadata, h.page_content[:50], '…')

## 6.01.6 `chunk_overlap` 설계 지침

| 용도 | chunk_size | chunk_overlap | 이유 |
|------|-----------|---------------|------|
| QA / FAQ | 300–500 토큰 | 10–15% | 질문이 한 chunk 안에 들어가야 recall ↑ |
| 요약 / 장문 RAG | 800–1500 토큰 | 10% | chunk 가 클수록 overlap 비중 ↓ 가능 |
| 멀티-홉 질의 | 300 토큰 | 20–25% | 경계 엔티티를 여러 chunk 가 공유 |
| 코드 / 구조화 문서 | `from_language` 로 위임 | — | `6.02` 참고 |

**첫 실험 기본값**: `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)`. 임베딩 모델의 `embedding_ctx_length`(OpenAI 3-small/large 는 8191 토큰) 한도 초과 금지.

## 체크리스트

- [ ] 일반 텍스트는 `RecursiveCharacterTextSplitter` 기본 선택
- [ ] 모델 토큰 한도에 맞출 땐 `from_tiktoken_encoder` 팩토리
- [ ] `chunk_overlap` 은 `chunk_size` 의 10–20% 로 출발
- [ ] 한국어는 separator 에 `". "` 추가 고려

## 다음

- `02_markdown_html_code.ipynb` — 구조 기반 분할 (Markdown/HTML/code)
- `03_semantic_chunker.ipynb` — 임베딩 거리 기반 의미 경계 탐지

## 참고

- 공식: https://docs.langchain.com/oss/python/concepts/text_splitters
- `langchain_text_splitters` 소스: https://github.com/langchain-ai/langchain/tree/master/libs/text-splitters
- tiktoken 인코딩: https://github.com/openai/tiktoken